I used `hanoi` v0.1.0

In [1]:
import hanoi 
import numpy as np
import math
from scipy.spatial.distance import squareform

A case with 3 loci, 10 individuals, 2 traits

In [2]:
L = 3
N = 10
n = 2

The mean effect, variance effect of allele 1 at the 3 loci on 2 traits are written as ndarray of shape (n, L)

In [3]:
M_m = np.array([[0.5, 1, 2],
                [0, 0, 0]])
M_v = np.array([[10, 10, 10],
                [1, 1, 1]])

The covariance effect of the 3 loci on 1 pair of traits are written as ndarray of shape (choose(n, 2), L)

In [4]:
M_c = np.array([[0, 1, 1]])

The MutEffect object is created

In [5]:
M = hanoi.MutEffect(mean = M_m, var = M_v, cov = M_c)

In [6]:
M.show()

Number of traits n:
2
Number of trait pairs:
1
Number of loci L:
3
Mutation effect of mean:
[[0.5 1.  2. ]
 [0.  0.  0. ]]
Mutation effect of variance:
[[10 10 10]
 [ 1  1  1]]
Mutation effect of covariance:
[[0 1 1]]


The genotype matrix is of shape (L, N) like VCF.

In [7]:
G_cur = np.zeros((L, N), dtype = int)

In [8]:
G_cur

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [9]:
G_cur[0, 0:3] = 1 # mutation at locus 0 in individuals 0-2
G_cur[1, 3:6] = 1 # mutation at locus 1 in individuals 3-5
G_cur[2, 6:9] = 1 # mutation at locus 2 in individuals 6-8

In [10]:
G_cur

array([[1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 1, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 1, 1, 0]])

Mean and covariance of reference genotype 0000 are written

In [11]:
mean_0 = np.array([0, 0]) # mean of 2 traits of genotype 00000...
cov_0 =  hanoi.cov_mtx([0,1],[0]) # variance covariance matrix of 2 traits

In [12]:
mean_0

array([0, 0])

In [13]:
cov_0

array([[0, 0],
       [0, 1]])

Within `hanoi.sim_generation()`, which is run within `hanoi.sim_generations()` , the following are done

First, breeding value is computed.

In [14]:
A = hanoi.comp_breed_val(mut_effect=M, genotype=G_cur)

In [15]:
A.show()

Number of traits:
 2
Number of trait pairs:
 1
Number of individuals:
 10
Breeding value of mean:
 [[0.25 0.  ]
 [0.25 0.  ]
 [0.25 0.  ]
 [0.5  0.  ]
 [0.5  0.  ]
 [0.5  0.  ]
 [1.   0.  ]
 [1.   0.  ]
 [1.   0.  ]
 [0.   0.  ]]
Breeding value of variance:
 [[5.  0.5]
 [5.  0.5]
 [5.  0.5]
 [5.  0.5]
 [5.  0.5]
 [5.  0.5]
 [5.  0.5]
 [5.  0.5]
 [5.  0.5]
 [0.  0. ]]
Breeding value of covariance:
 [[0. ]
 [0. ]
 [0. ]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0. ]]
Covariance matrices of breeding values :
 [[[5.  0. ]
  [0.  0.5]]

 [[5.  0. ]
  [0.  0.5]]

 [[5.  0. ]
  [0.  0.5]]

 [[5.  0.5]
  [0.5 0.5]]

 [[5.  0.5]
  [0.5 0.5]]

 [[5.  0.5]
  [0.5 0.5]]

 [[5.  0.5]
  [0.5 0.5]]

 [[5.  0.5]
  [0.5 0.5]]

 [[5.  0.5]
  [0.5 0.5]]

 [[0.  0. ]
  [0.  0. ]]]


In [16]:
z = hanoi.sim_pheno(breed_val=A, mean_0=mean_0, cov_0 = cov_0)

In [17]:
z

array([[-0.41008784,  1.59326512],
       [-0.87209425,  3.75487354],
       [ 0.61002266, -0.55254468],
       [ 0.95131857, -0.11162177],
       [ 0.72172615,  0.04748976],
       [-3.08660878, -1.45749408],
       [ 7.36752525,  2.93874793],
       [-2.59403829,  2.3700744 ],
       [ 1.01667238, -1.10118188],
       [        nan,         nan]])

This is done recursively over generations in `hanoi.sim_generations()`

In [18]:
generations = hanoi.sim_generations(n_gen = 0, 
                                    mut_effect = M, 
                                    genotype = G_cur,
                                    mut_rate = 0, 
                                    fit_func = "fit_neutral", 
                                    mean_0 = mean_0,
                                    cov_0= cov_0, 
                                   )

The number of generations can be obtained as the `n_gen` attribute

In [19]:
generations.n_gen

4

The initial and last allele frequencies can be obtained using `allele_freq_first()` and `allele_freq_last()`

In [20]:
generations.allele_freq_first()

array([0.15, 0.15, 0.15])

In [21]:
generations.allele_freq_last()

array([0., 0., 0.])

The allele frequency trajectry can be obtained with `allele_freqs()` method.
It gives allele frequency trajectory before selection for each generation.

In [23]:
generations.allele_freqs()

array([[0.15, 0.15, 0.15],
       [0.1 , 0.2 , 0.1 ],
       [0.15, 0.1 , 0.25],
       [0.25, 0.15, 0.2 ],
       [0.2 , 0.35, 0.15],
       [0.2 , 0.25, 0.25],
       [0.2 , 0.1 , 0.4 ],
       [0.2 , 0.25, 0.6 ],
       [0.2 , 0.3 , 0.55],
       [0.25, 0.3 , 0.5 ],
       [0.25, 0.35, 0.55],
       [0.15, 0.5 , 0.45],
       [0.1 , 0.65, 0.45],
       [0.1 , 0.6 , 0.4 ],
       [0.05, 0.55, 0.55],
       [0.05, 0.8 , 0.55],
       [0.  , 0.75, 0.7 ],
       [0.  , 0.75, 0.55],
       [0.  , 0.55, 0.65],
       [0.  , 0.45, 0.65],
       [0.  , 0.45, 0.45],
       [0.  , 0.5 , 0.7 ],
       [0.  , 0.45, 0.75],
       [0.  , 0.55, 0.5 ],
       [0.  , 0.75, 0.65],
       [0.  , 0.85, 0.65],
       [0.  , 0.7 , 0.65],
       [0.  , 0.8 , 0.5 ],
       [0.  , 0.85, 0.5 ],
       [0.  , 0.8 , 0.4 ],
       [0.  , 0.9 , 0.5 ],
       [0.  , 0.8 , 0.4 ],
       [0.  , 0.75, 0.3 ],
       [0.  , 0.75, 0.25],
       [0.  , 0.7 , 0.1 ],
       [0.  , 0.75, 0.25],
       [0.  , 0.55, 0.15],
 

In the last line, the allele is not fixed. This is because allele fixes after selection, while this method returns before selection.

To add the frequency after selection of the last generation,

In [24]:
np.append(generations.allele_freqs(), generations.allele_freq_last()[np.newaxis,:], axis = 0)

array([[0.15, 0.15, 0.15],
       [0.1 , 0.2 , 0.1 ],
       [0.15, 0.1 , 0.25],
       [0.25, 0.15, 0.2 ],
       [0.2 , 0.35, 0.15],
       [0.2 , 0.25, 0.25],
       [0.2 , 0.1 , 0.4 ],
       [0.2 , 0.25, 0.6 ],
       [0.2 , 0.3 , 0.55],
       [0.25, 0.3 , 0.5 ],
       [0.25, 0.35, 0.55],
       [0.15, 0.5 , 0.45],
       [0.1 , 0.65, 0.45],
       [0.1 , 0.6 , 0.4 ],
       [0.05, 0.55, 0.55],
       [0.05, 0.8 , 0.55],
       [0.  , 0.75, 0.7 ],
       [0.  , 0.75, 0.55],
       [0.  , 0.55, 0.65],
       [0.  , 0.45, 0.65],
       [0.  , 0.45, 0.45],
       [0.  , 0.5 , 0.7 ],
       [0.  , 0.45, 0.75],
       [0.  , 0.55, 0.5 ],
       [0.  , 0.75, 0.65],
       [0.  , 0.85, 0.65],
       [0.  , 0.7 , 0.65],
       [0.  , 0.8 , 0.5 ],
       [0.  , 0.85, 0.5 ],
       [0.  , 0.8 , 0.4 ],
       [0.  , 0.9 , 0.5 ],
       [0.  , 0.8 , 0.4 ],
       [0.  , 0.75, 0.3 ],
       [0.  , 0.75, 0.25],
       [0.  , 0.7 , 0.1 ],
       [0.  , 0.75, 0.25],
       [0.  , 0.55, 0.15],
 

Breeding value can be retrieved with `.breed_val` attribute.

In [25]:
generations.breed_val

array([<hanoi.hanoi.BreedVal object at 0x761b700e75f0>,
       <hanoi.hanoi.BreedVal object at 0x761b6485f860>], dtype=object)

To obtain breeding value of generation 5

In [26]:
generations.breed_val[5].show()

Number of traits:
 2
Number of trait pairs:
 1
Number of individuals:
 10
Breeding value of mean:
 [[1.25 0.  ]
 [2.   0.  ]
 [0.   0.  ]
 [0.75 0.  ]
 [0.5  0.  ]
 [1.5  0.  ]
 [1.5  0.  ]
 [0.5  0.  ]
 [0.   0.  ]
 [0.5  0.  ]]
Breeding value of variance:
 [[10.   1. ]
 [10.   1. ]
 [ 0.   0. ]
 [10.   1. ]
 [ 5.   0.5]
 [10.   1. ]
 [10.   1. ]
 [ 5.   0.5]
 [ 0.   0. ]
 [10.   1. ]]
Breeding value of covariance:
 [[0.5]
 [1. ]
 [0. ]
 [0.5]
 [0.5]
 [1. ]
 [1. ]
 [0.5]
 [0. ]
 [0. ]]
Covariance matrices of breeding values :
 [[[10.   0.5]
  [ 0.5  1. ]]

 [[10.   1. ]
  [ 1.   1. ]]

 [[ 0.   0. ]
  [ 0.   0. ]]

 [[10.   0.5]
  [ 0.5  1. ]]

 [[ 5.   0.5]
  [ 0.5  0.5]]

 [[10.   1. ]
  [ 1.   1. ]]

 [[10.   1. ]
  [ 1.   1. ]]

 [[ 5.   0.5]
  [ 0.5  0.5]]

 [[ 0.   0. ]
  [ 0.   0. ]]

 [[10.   0. ]
  [ 0.   1. ]]]


Or

In [27]:
generations.generation(5).breed_val.show()

Number of traits:
 2
Number of trait pairs:
 1
Number of individuals:
 10
Breeding value of mean:
 [[1.25 0.  ]
 [2.   0.  ]
 [0.   0.  ]
 [0.75 0.  ]
 [0.5  0.  ]
 [1.5  0.  ]
 [1.5  0.  ]
 [0.5  0.  ]
 [0.   0.  ]
 [0.5  0.  ]]
Breeding value of variance:
 [[10.   1. ]
 [10.   1. ]
 [ 0.   0. ]
 [10.   1. ]
 [ 5.   0.5]
 [10.   1. ]
 [10.   1. ]
 [ 5.   0.5]
 [ 0.   0. ]
 [10.   1. ]]
Breeding value of covariance:
 [[0.5]
 [1. ]
 [0. ]
 [0.5]
 [0.5]
 [1. ]
 [1. ]
 [0.5]
 [0. ]
 [0. ]]
Covariance matrices of breeding values :
 [[[10.   0.5]
  [ 0.5  1. ]]

 [[10.   1. ]
  [ 1.   1. ]]

 [[ 0.   0. ]
  [ 0.   0. ]]

 [[10.   0.5]
  [ 0.5  1. ]]

 [[ 5.   0.5]
  [ 0.5  0.5]]

 [[10.   1. ]
  [ 1.   1. ]]

 [[10.   1. ]
  [ 1.   1. ]]

 [[ 5.   0.5]
  [ 0.5  0.5]]

 [[ 0.   0. ]
  [ 0.   0. ]]

 [[10.   0. ]
  [ 0.   1. ]]]


Fitness of individuals over generations can be obtained

In [28]:
generations.fitness

array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 

The breeding value of mean effect of trait 0 over generations

In [29]:
np.array([breed_val.mean[:, 0] for breed_val in generations.breed_val]) + mean_0[0]

array([[0.25, 0.25, 0.25, 0.5 , 0.5 , 0.5 , 1.  , 1.  , 1.  , 0.  ],
       [0.25, 0.  , 0.  , 0.25, 1.  , 1.  , 0.5 , 0.5 , 1.  , 0.  ],
       [0.  , 0.  , 1.25, 0.  , 1.  , 0.25, 2.  , 1.25, 0.5 , 0.5 ],
       [0.5 , 0.25, 1.  , 1.  , 0.25, 0.75, 0.5 , 0.  , 1.25, 1.25],
       [0.  , 1.  , 1.  , 1.  , 0.25, 1.5 , 0.75, 0.5 , 0.25, 1.25],
       [1.25, 2.  , 0.  , 0.75, 0.5 , 1.5 , 1.5 , 0.5 , 0.  , 0.5 ],
       [0.  , 1.5 , 2.5 , 0.  , 0.5 , 1.25, 1.  , 1.  , 1.  , 1.25],
       [2.  , 2.  , 1.75, 1.5 , 2.  , 1.  , 1.25, 1.25, 1.5 , 1.25],
       [0.  , 2.25, 1.75, 1.25, 2.5 , 1.25, 1.5 , 1.5 , 1.  , 2.  ],
       [1.5 , 1.25, 0.5 , 2.  , 1.5 , 2.  , 2.25, 1.5 , 1.5 , 0.25],
       [1.25, 2.5 , 2.25, 1.  , 1.25, 2.  , 0.25, 1.5 , 2.5 , 1.25],
       [1.75, 1.5 , 1.  , 1.75, 2.5 , 3.  , 0.5 , 0.5 , 1.5 , 0.75],
       [2.25, 2.5 , 1.5 , 0.5 , 1.5 , 1.75, 0.5 , 3.  , 0.5 , 2.  ],
       [3.  , 0.5 , 0.5 , 1.25, 1.5 , 2.5 , 1.25, 1.5 , 2.  , 0.5 ],
       [2.25, 1.5 , 1.5 , 0.  , 3.

In [30]:
np.array([breed_val.var[:, 0] for breed_val in generations.breed_val]) + cov_0[0,0]

array([[ 5.,  5.,  5.,  5.,  5.,  5.,  5.,  5.,  5.,  0.],
       [ 5.,  0.,  0.,  5.,  5., 10.,  5.,  5.,  5.,  0.],
       [ 0.,  0., 10.,  0.,  5.,  5., 10., 10.,  5.,  5.],
       [ 5.,  5.,  5.,  5.,  5., 10.,  5.,  0., 10., 10.],
       [ 0.,  5., 10., 10.,  5., 10., 10.,  5.,  5., 10.],
       [10., 10.,  0., 10.,  5., 10., 10.,  5.,  0., 10.],
       [ 0., 15., 15.,  0.,  5., 10.,  5.,  5.,  5., 10.],
       [10., 10., 15., 10., 10., 10., 10., 10., 10., 10.],
       [ 0., 15., 15., 10., 15., 10., 10., 10., 10., 10.],
       [15., 10.,  5., 15., 10., 10., 15., 10., 10.,  5.],
       [10., 15., 15., 10., 10., 15.,  5., 10., 15., 10.],
       [15., 10.,  5., 15., 15., 20.,  5.,  5., 10., 10.],
       [20., 15., 10.,  5., 10., 15.,  5., 20.,  5., 15.],
       [20.,  5.,  5., 15., 10., 15., 10., 10., 15.,  5.],
       [20., 10., 10.,  0., 20., 10., 15., 15., 15.,  0.],
       [10., 15., 15., 20., 10., 15.,  5., 15., 15., 20.],
       [ 5., 15., 15., 10., 15., 20., 20., 15., 15., 15.

In [31]:
generations.generation(1).phenotype[:,0]

array([-3.33810468,  0.        ,  0.        , -2.93838351,  0.19595215,
        0.70686594, -2.04952978,  0.5060604 ,  2.54918204,  0.        ])

In [32]:
generations.phenotype[:, :, 0]

array([[-1.88360587e+00,  1.39457525e+00,  2.36756505e+00,
        -1.79411343e+00, -2.66491934e-01, -1.48137997e+00,
         3.83956176e+00, -3.80648558e-01,  2.52149717e+00,
         0.00000000e+00],
       [-3.33810468e+00,  0.00000000e+00,  0.00000000e+00,
        -2.93838351e+00,  1.95952147e-01,  7.06865938e-01,
        -2.04952978e+00,  5.06060404e-01,  2.54918204e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  9.45802902e-01,
         0.00000000e+00,  3.53227907e+00,  1.55388203e+00,
        -1.41594928e+00,  3.44126609e+00,  1.35804039e+00,
        -1.70757591e+00],
       [ 7.02960140e-01, -1.15313279e+00, -1.13282154e+00,
         7.51761519e-01, -6.16526910e-01, -4.50444068e+00,
        -3.48178945e+00,  0.00000000e+00, -8.45942778e-01,
         1.54481790e+00],
       [ 0.00000000e+00,  4.76059642e-01,  2.70592727e+00,
         2.01925438e+00, -4.53957216e-01, -2.82684465e+00,
         9.78272795e-01,  1.71696297e+00, -7.48920903e-01,
         5.

The distribution of number of offspring over generations can be obtained

In [33]:
generations.n_offspring

array([[2, 1, 2, 3, 2, 2, 0, 4, 2, 2],
       [1, 0, 2, 3, 5, 1, 1, 2, 3, 2],
       [0, 1, 0, 3, 3, 3, 2, 3, 3, 2],
       [3, 0, 3, 2, 3, 0, 6, 0, 1, 2],
       [3, 1, 1, 1, 3, 3, 2, 1, 2, 3],
       [5, 3, 2, 0, 2, 1, 2, 1, 3, 1],
       [2, 3, 1, 0, 4, 1, 1, 4, 1, 3],
       [1, 1, 3, 5, 4, 0, 2, 2, 2, 0],
       [1, 4, 1, 2, 0, 5, 1, 1, 3, 2],
       [2, 1, 5, 2, 0, 2, 1, 1, 2, 4],
       [2, 1, 2, 3, 2, 5, 0, 1, 2, 2],
       [1, 0, 1, 1, 6, 3, 3, 2, 1, 2],
       [2, 1, 2, 2, 3, 2, 1, 3, 2, 2],
       [4, 1, 1, 1, 2, 3, 1, 4, 1, 2],
       [2, 1, 3, 0, 4, 2, 3, 3, 1, 1],
       [3, 0, 4, 1, 2, 2, 1, 1, 4, 2],
       [4, 3, 5, 1, 1, 1, 2, 1, 1, 1],
       [1, 2, 3, 1, 3, 2, 2, 4, 2, 0],
       [2, 1, 5, 4, 1, 4, 0, 0, 1, 2],
       [0, 2, 2, 3, 3, 5, 1, 1, 2, 1],
       [2, 2, 1, 1, 4, 5, 1, 2, 0, 2],
       [1, 4, 4, 1, 2, 2, 1, 1, 2, 2],
       [1, 1, 0, 3, 2, 4, 5, 0, 2, 2],
       [2, 2, 3, 0, 5, 1, 2, 1, 3, 1],
       [2, 2, 3, 2, 2, 0, 3, 2, 1, 3],
       [1, 2, 4, 0, 1, 3,

If the population go extinct by too strong selection, genotype and allele frequency are filled by np.nan

In [34]:

generations = hanoi.sim_generations(n_gen =0, 
                                    mut_effect = M, 
                                    genotype = G_cur,
                                    mut_rate = 0, 
                                    fit_func = "fit_step", 
                                    boxes = np.array([[[100], [math.inf]]]), 
                                    mean_0 = mean_0,
                                    cov_0= cov_0, 
                                   )

In [35]:
generations.genotype_next[-1]

array([[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]])

In [36]:
generations.allele_freq_last()

array([nan, nan, nan])

In [37]:
M.show()

Number of traits n:
2
Number of trait pairs:
1
Number of loci L:
3
Mutation effect of mean:
[[0.5 1.  2. ]
 [0.  0.  0. ]]
Mutation effect of variance:
[[10 10 10]
 [ 1  1  1]]
Mutation effect of covariance:
[[0 1 1]]


In [38]:
res = []
for i in range(10):
    sim = hanoi.sim_generations(n_gen =0, 
                                mut_effect = M, 
                                genotype = G_cur, 
                                mut_rate = 0, 
                                fit_func = "fit_step", 
                                boxes = np.array([[[1], [math.inf]]]),
                                mean_0 = np.array([0]),
                                cov_0 = np.array([1])
                               )
    res.append([sim.n_gen, sim.allele_freq_last()[0]])

In [39]:
np.array(res)

array([[ 4.,  1.],
       [ 2., nan],
       [ 1., nan],
       [ 4.,  0.],
       [ 1., nan],
       [ 1., nan],
       [ 5.,  1.],
       [ 4., nan],
       [ 2.,  0.],
       [ 8., nan]])

In [40]:
sim.fitness

array([[0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0., 1., 0., 0., 1., 1.],
       [1., 0., 0., 0., 1., 0., 0., 0., 1., 1.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])